In [12]:
#rfm değişkenlerini çıkarma kısmı
import os, pandas as pd

# Proje köküne geç
os.chdir(r"C:\Users\emine\Desktop\rfm-xai-segmentation")

# Veriyi tekrar oku
df = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [13]:
# Eksik müşteri ID'leri çıkar
df = df.dropna(subset=["Customer ID"])

# İade faturaları çıkar (Invoice 'C' ile başlıyorsa)
df = df[~df["Invoice"].astype(str).str.startswith("C")]

# Negatif veya sıfır miktar/fiyatları çıkar
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

# Tarih ve toplam fiyat sütunu oluştur
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["Price"]

df.shape


(397885, 9)

In [14]:
import datetime as dt

snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days=1)

rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,  # Recency
    "Invoice": "nunique",                                     # Frequency
    "TotalPrice": "sum"                                       # Monetary
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]
rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


In [19]:
import os
os.makedirs("data/processed", exist_ok=True)

# hem CSV hem Parquet kaydedelim (CSV'yi clustering okuyacak)
rfm.to_csv("data/processed/rfm_segmented.csv", index=False)
#rfm.to_parquet("data/processed/rfm_segmented.parquet", index=False)

rfm.shape


(4338, 4)